In [ ]:
# Bootstrap: i notebook vivono in finetuning/ ma tutto il progetto (import,
# data/, results/finetuning/, main.py) e' relativo alla root del repo -> ci spostiamo li'.
import os, sys
from pathlib import Path

if not Path("main.py").exists():
    os.chdir(Path.cwd().parent)
assert Path("main.py").exists(), "Esegui il notebook dalla root del repo o da finetuning/"
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print(f"Working directory: {Path.cwd()}")

# 3 Zero-shot vs fine-tuned

For the final piece of this project, it's time to compare the two models

In [ ]:
from pathlib import Path

from finetuning.evaluator import find_checkpoints, load_results

DATA_DIR = Path("data/loveda")
baseline = load_results("results/finetuning/baseline_generico.json")

print("list of checkpoints:")
for p in find_checkpoints():
    print(f"  {p}")

In [ ]:
# two candidates: best and last
# best could have masked attention still on
# last have unmasked
ckpts = [p for p in find_checkpoints() if "loveda_ft_da_coco" in str(p)]
BEST_CKPT = next(p for p in ckpts if p.name.startswith("best"))
LAST_CKPT = next(p for p in ckpts if p.name == "last.ckpt")
print(f"best: {BEST_CKPT}\nlast: {LAST_CKPT}")

## 1. Training Curves

In [ ]:
from finetuning.visualize import plot_training_curves

plot_training_curves("loveda_ft_da_coco", save_path="results/finetuning/ft_da_coco/curve_training.png")
if Path("lightning_logs/loveda_ft_da_dinov2").exists():
    plot_training_curves("loveda_ft_da_dinov2", save_path="results/finetuning/ft_da_dinov2/curve_training.png");

## 2. Evaluation of the two fine-tuned models (best vs last)

In [ ]:
from datasets.loveda_semantic import LoveDASemantic
from finetuning.evaluator import (
    build_eomt_small, load_network_weights, evaluate_on_loveda, save_results,
)

dm = LoveDASemantic(path=str(DATA_DIR), batch_size=2, num_workers=2).setup()

candidates = {}
for tag, ckpt in [("best", BEST_CKPT), ("last", LAST_CKPT)]:
    net = build_eomt_small(num_classes=7)
    net = load_network_weights(net, ckpt)
    print(f"\n--- {tag}: {ckpt.name} ---")
    candidates[tag] = (net, evaluate_on_loveda(net, dm, class_mapping=None))

winner = max(candidates, key=lambda tag: candidates[tag][1]["miou"])
ft_network, ft_results = candidates[winner]
ft_results["checkpoint"] = str({"best": BEST_CKPT, "last": LAST_CKPT}[winner])
print(f"\Winner is: {winner} (mIoU {ft_results['miou'] * 100:.2f})")
save_results(ft_results, "results/finetuning/finetuned_da_coco.json")

In [ ]:
# same code, for the ablation test
abl_network = None
abl_ckpts = [p for p in find_checkpoints() if "loveda_ft_da_dinov2" in str(p)]
if abl_ckpts:
    abl_candidates = {}
    for tag in ["best", "last"]:
        ckpt = next(p for p in abl_ckpts if p.name.startswith(tag))
        net = build_eomt_small(num_classes=7)
        net = load_network_weights(net, ckpt)
        print(f"\n--- ablation {tag}: {ckpt.name} ---")
        abl_candidates[tag] = (net, evaluate_on_loveda(net, dm, class_mapping=None))

    tag = max(abl_candidates, key=lambda t: abl_candidates[t][1]["miou"])
    abl_network, abl_results = abl_candidates[tag]
    abl_results["checkpoint"] = str(next(p for p in abl_ckpts if p.name.startswith(tag)))
    print(f"\nablation: {tag} (mIoU {abl_results['miou'] * 100:.2f})")
    save_results(abl_results, "results/finetuning/finetuned_da_dinov2.json")
else:
    print("no ablation")

## 3. Comparing results

In [ ]:
import pandas as pd

from finetuning.loveda import CLASS_NAMES

results_by_name = {
    "Generic (COCO, zero-shot)": baseline,
    "Fine-tuned (COCO → LoveDA)": ft_results,
}

# if abaltion existss
abl_path = Path("results/finetuning/finetuned_da_dinov2.json")
if abl_path.exists():
    results_by_name["Fine-tuned (only DINOv2)"] = load_results(abl_path)

table = pd.DataFrame({
    name: {cls: res["iou_per_class"][cls] * 100 for cls in CLASS_NAMES}
    for name, res in results_by_name.items()
}).round(2)
table.loc["— mIoU —"] = [res["miou"] * 100 for res in results_by_name.values()]
if len(results_by_name) >= 2:
    cols = list(results_by_name)
    table["delta (FT − generic)"] = (table[cols[1]] - table[cols[0]]).round(2)
display(table)
table.to_csv("results/finetuning/confronto.csv")

In [ ]:
from finetuning.visualize import plot_comparison_bars

plot_comparison_bars(results_by_name, save_path="results/finetuning/confronto_per_classe.png")

In [ ]:
from finetuning.evaluator import compute_confusion_matrix
from finetuning.visualize import plot_confusion_matrix

models_cm = {"ft_da_coco": ft_network}
if abl_network is not None:
    models_cm["ft_da_dinov2"] = abl_network

for tag, net in models_cm.items():
    cm = compute_confusion_matrix(net, dm, class_mapping=None)
    plot_confusion_matrix(cm, title=f"{tag} — confusion on LoveDA val",
                          save_path=f"results/finetuning/{tag}/confusione.png")

## 4. Image comparison

In [ ]:
import json

from finetuning.coco_classes import build_coco_to_loveda_mapping
from finetuning.evaluator import predict
from finetuning.loveda import CLASS_NAMES
from finetuning.visualize import get_val_samples, show_samples

SAMPLE_INDICES = json.loads(Path("results/finetuning/sample_indices.json").read_text())
imgs, targets = get_val_samples(dm, SAMPLE_INDICES)

generic = build_eomt_small(num_classes=133)
generic = load_network_weights(generic, "checkpoints/coco_panoptic_eomt_small_640_2x.bin")

preds = {
    "Generic (zero-shot)": predict(
        generic.cuda(), imgs, class_mapping=build_coco_to_loveda_mapping(CLASS_NAMES)
    ),
    "FT da COCO": predict(ft_network.cuda(), imgs, class_mapping=None),
}
if abl_network is not None:
    preds["FT da DINOv2"] = predict(abl_network.cuda(), imgs, class_mapping=None)

show_samples(imgs, targets, preds, save_path="results/finetuning/qualitative_confronto.png")